# 08 - second arm: front heads on frozen features

**GPU runtime, high RAM.** Recorded before the run (2026-09-25; see README, Protocol): the test
run showed frozen linear probes fail at calving fronts. Four heads on the same
frozen features decide whether that is fixable and by what:

| row | head | learned |
|---|---|---|
| 1 | linear probe on the 16 px grid (arm 1, notebooks 03-07, done) | ~3 k |
| 2 | AnyUp -> per-pixel features -> linear probe | ~3 k |
| 3 | grid -> small learned decoder, image as skip input | ~1-2 M |
| 4 | AnyUp -> small learned decoder | ~1-2 M |

The counts are estimates recorded before the run; step 2 below has the real
counts and rows 5-6, added later.

**Rules:** every choice is made on the validation glacier SI; the test set is run
once at the end for the finished heads. Nothing changes after test numbers.

**Step 1 (this cell): which layer to tap.** So far every probe used the last
layer. Intermediate layers often keep finer spatial detail, which matters for a
thin front. Criterion: 3-class zone patch mIoU on SI (cheap proxy, declared).

In [1]:
# --- cell: mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# --- cell: secrets, GPU, data, AnyUp weights check ---
import os, shutil
from pathlib import Path
import torch
from sartransfer.env import load_env, require, model_cache_on_drive
from sartransfer.data import inventory as inv
from sartransfer.models.anyup_loader import load_anyup, fetch_anyup_weights

load_env()
require("HF_TOKEN", "gated model weights")
model_cache_on_drive()          # weights downloaded once, kept on Drive
assert torch.cuda.is_available(), "switch to a GPU runtime"
print("gpu:", torch.cuda.get_device_properties(0).name)
DRIVE = Path("/content/drive/MyDrive")
RAW, RES = DRIVE / "sar-transfer" / "raw", DRIVE / "sar-transfer" / "results"
ANYUP_W = DRIVE / "sar-transfer" / "weights" / "anyup_multi_backbone.pth"
inv.unzip_caffe(inv.fetch_caffe_zip(RAW), Path("/content/caffe"))
DATA = inv.find_data_root(Path("/content/caffe"))
df = inv.scan(DATA, subdirs=("sar_images",))
anyup = load_anyup(fetch_anyup_weights(ANYUP_W))   # fetched once to Drive, hash-checked
print(len(df), "SAR images | AnyUp weights OK | free disk:",
      f"{shutil.disk_usage('/content').free/1e9:.0f} GB")

In [4]:
# --- cell: STEP 1 -- layer sweep on SI (13-14 min per encoder, recorded) ---
import time
import pandas as pd
from sartransfer.runs import layer_sweep

t0, results = time.perf_counter(), {}
for encoder, norm in (("cradio-v4-h", "caffe"), ("dinov3-l-sat", "caffe")):      # photo DINOv3 replaced by SAT-493M (2026-09-25)
    try:
        r = layer_sweep(encoder, norm, df, DATA, RES, token=os.environ["HF_TOKEN"])
        results[encoder] = "skipped (already saved)" if r is None else "done"
    except Exception as e:
        results[encoder] = f"FAILED: {type(e).__name__}: {str(e)[:300]}"
        print(results[encoder])
print(f"\nfinished in {(time.perf_counter()-t0)/60:.0f} min:", results)

import glob
sw = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(str(RES / "layersweep__*.csv")))])
pd.set_option("display.width", 200, "display.precision", 3)
print("\n=== SI patch mIoU per tapped layer (band=all) ===")
print(sw[(sw.grouping == "band") & (sw.group == "all")].pivot_table(index="run", columns="layer", values="mIoU").round(3))
print("\n=== per band, best vs last layer ===")
b = sw[sw.grouping == "band"].pivot_table(index=["run", "layer"], columns="group", values="mIoU").round(3)
print(b)

skip cradio-v4-h__caffe__tile512: layer sweep already on Drive

===== layer sweep dinov3-l-sat__caffe__tile512 =====


config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

dinov3-l-sat: 303.1 M parameters loaded
24 blocks; tapping layers [6, 12, 18, 21, 24]
restricted to splits ('train',): 438 of 681 images
trained 5 probes on 953,371 patches, 11.6 min
restricted to splits ('val',): 121 of 681 images
SI patch mIoU per layer: {6: 0.2428, 12: 0.4296, 18: 0.5319, 21: 0.5673, 24: 0.5397} | best layer 21 | 13.3 min

finished in 13 min: {'cradio-v4-h': 'skipped (already saved)', 'dinov3-l-sat': 'done'}

=== SI patch mIoU per tapped layer (band=all) ===
layer                              6      8      12     16     18     21     24     28     32
run                                                                                          
cradio-v4-h__caffe__tile512       NaN  0.352    NaN  0.496    NaN    NaN  0.442  0.426  0.587
dinov3-l-photo__caffe__tile512  0.251    NaN  0.539    NaN  0.585  0.581  0.548    NaN    NaN
dinov3-l-sat__caffe__tile512    0.243    NaN  0.430    NaN  0.532  0.567  0.540    NaN    NaN

=== per band, best vs last layer ===
group    

## Reading step 1

- The last layer is what notebooks 03-07 used. If an earlier layer is clearly better
  on SI (more than ~0.01), the heads in step 2 tap that layer; otherwise they
  keep the last one. The choice is fixed in advance, before step 2 runs.
- Result: C-RADIO keeps its last layer (32); DINOv3 SAT taps layer 21 (0.567 vs
  0.540 at 24); photo DINOv3 layer 18. Step 2 below runs only C-RADIO and DINOv3
  SAT, at these layers (row 5: four layers each); the photo DINOv3 rows (layer 18)
  in its SI table are saved CSVs from an earlier session, before that model was
  replaced (2026-09-25).

## Step 2: the heads (layers fixed by step 1: C-RADIO **32**; DINOv3 SAT-493M **21**; the photo DINOv3, layer 18, was replaced 2026-09-25)

Per encoder, one in-memory extraction at the chosen layer, then:

| row | head | trainable | how AnyUp is used |
|---|---|---|---|
| 1 | linear probe, logits upsampled bilinearly | ~4 k | – |
| 2 | linear probe on AnyUp-upsampled pixels | ~4 k | trained on 240 k sampled pixels; scored **exactly** by upsampling the 4 logits (identical to probing upsampled features) |
| 3 | small decoder with image skip | 0.54 M (DINOv3) / 0.57 M (C-RADIO) | – |
| 4 | learned 1×1 reduction → AnyUp attention → light head | 0.11 / 0.13 M | attention from frozen features, applied to the learned values |
| 5 | row 3's decoder on **four layers** concatenated (every quarter of the depth) | 0.84 / 0.94 M | – |
| 6 | 1×1 reduction → AnyUp to **1/4 resolution** → row 3's last three stages | 0.24 / 0.26 M | as row 4, but 16x cheaper and keeping the multi-scale image skip |

Rows 4 and 6: the logs below print more (166.6 k / 183.0 k and 281.3 k / 305.9 k);
the extra parameters are unused image-stem levels that never got a gradient
(see docs/results.md).

Rows 5-6 were added 2026-09-25 after reading the rows 1-4 SI table (recorded
before the run; see README, Protocol); the test set is still untouched. Row 3 is also trained with
seeds 1 and 2 (same tiles, other initialisation and tile order) so the test
reports a spread, not one lucky run.

Rows 3-6 train 4 000 steps on up to 4 000 training tiles (row 5: 2 000), no checkpoint picked
on SI. Everything is scored through the same zone → front → MDE route as
notebooks 06-07 ("step 8" and "step 9" in the printed references) and saved
to Drive (`heads_val_*`), with the trained heads
(`heads__*.pt`) for the one-time test run in the last cell.

In [5]:
# --- cell: STEP 2 on SI -- train + score the heads; scored rows are skipped (recorded: 51-110 min per run, about 4.8 h in total) ---
import glob, time
import pandas as pd
from sartransfer.heads import run_heads, summary_table

LAYER = {"dinov3-l-sat": 21, "cradio-v4-h": 32}         # from step 1 (best SI patch mIoU of the sweep), fixed in advance
TAPS = {"dinov3-l-sat": (6, 12, 18, 24), "cradio-v4-h": (8, 16, 24, 32)}   # row 5: every quarter of the depth
for encoder, norm in [(e, "caffe") for e in LAYER]:
    t0 = time.perf_counter()
    try:
        run_heads(encoder, norm, LAYER[encoder], df, DATA, RES, token=os.environ["HF_TOKEN"],
                  anyup=anyup, split="val", rows=(1, 2, 3, 4, 6))
        run_heads(encoder, norm, TAPS[encoder], df, DATA, RES, token=os.environ["HF_TOKEN"],
                  anyup=anyup, split="val", rows=(5,), train_frac=0.01, max_train_tiles=2000)   # 4x wider features: 2 000 tiles (printed: RAM used 67 / 80 of 179 GB)
        run_heads(encoder, norm, LAYER[encoder], df, DATA, RES, token=os.environ["HF_TOKEN"],
                  anyup=anyup, split="val", rows=(3,), seeds=(1, 2), train_frac=0.01)   # row 3 spread
    except Exception as e:
        print(f"FAILED {encoder}: {type(e).__name__}: {str(e)[:300]}")
    print(f"{encoder}: {(time.perf_counter() - t0) / 60:.0f} min\n")

si = summary_table(RES, "val")          # rebuilt from the per-row CSVs: complete across sessions
pd.set_option("display.width", 200, "display.precision", 3)
print("=== SI: MDE (m, lower is better) and 4-class pixel mIoU per head ===")
print(si[["run", "row", "seed", "head", "params", "MDE_m", "no_front", "mIoU4_pixel", "minutes"]].round(3).to_string(index=False))
print("\nreference on SI (step 8): U-Net 4-class 1,732 m / 0 no-front; C-RADIO probe last layer 1,816 m; DINOv3 probe last layer 2,018 m")


===== heads dinov3-l-sat__caffe__tile512__L21 on val: rows [1, 2, 3, 4, 6], seeds [0] =====


Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

dinov3-l-sat: 303.1 M parameters loaded
restricted to splits ('train', 'val'): 559 of 681 images
  50/559  0.3 min elapsed, 0.39 s/image, ETA 3.3 min
  100/559  0.6 min elapsed, 0.34 s/image, ETA 2.6 min
  150/559  0.7 min elapsed, 0.28 s/image, ETA 1.9 min
  200/559  1.2 min elapsed, 0.36 s/image, ETA 2.1 min
  250/559  1.8 min elapsed, 0.42 s/image, ETA 2.2 min
  300/559  2.3 min elapsed, 0.47 s/image, ETA 2.0 min
  350/559  2.9 min elapsed, 0.50 s/image, ETA 1.7 min
  400/559  3.2 min elapsed, 0.48 s/image, ETA 1.3 min
  450/559  3.4 min elapsed, 0.45 s/image, ETA 0.8 min
  500/559  3.5 min elapsed, 0.42 s/image, ETA 0.4 min
  550/559  3.8 min elapsed, 0.41 s/image, ETA 0.1 min
  559/559  3.9 min elapsed, 0.42 s/image, ETA 0.0 min
in memory: train X (2148307, 1024) (4.4 GB fp16), 121 val images
features ready in 3.9 min | 4000 train tiles kept | RAM used 51 of 179 GB
  row 1 linear-grid           4.1 k params | MDE   1853 m (2 no front) | 4-class pixel mIoU 0.694 | 2.3 min


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:2954: UserWarning: Mismatch dtype between input and weight: input dtype = c10::BFloat16, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at /pytorch/aten/src/ATen/native/layer_norm.cpp:344.)
  return torch.rms_norm(input, normalized_shape, weight, eps)


row 2: probe on 240,000 upsampled pixels from 600 tiles
  row 2 anyup-linear          4.1 k params | MDE   1849 m (0 no front) | 4-class pixel mIoU 0.678 | 8.8 min
    step  1000  loss 0.3470  1.2 min
    step  2000  loss 0.1675  2.4 min
    step  3000  loss 0.1372  3.6 min
    step  4000  loss 0.1204  4.9 min
  row 3 grid-decoder        544.4 k params | MDE   1301 m (0 no front) | 4-class pixel mIoU 0.760 | 6.2 min
    step  1000  loss 0.3644  7.1 min
    step  2000  loss 0.1884  14.2 min
    step  3000  loss 0.1527  21.3 min
    step  4000  loss 0.1380  28.4 min
  row 4 anyup-decoder       166.6 k params | MDE   1402 m (0 no front) | 4-class pixel mIoU 0.757 | 33.1 min
    step  1000  loss 0.3161  3.5 min
    step  2000  loss 0.1559  7.0 min
    step  3000  loss 0.1248  10.4 min
    step  4000  loss 0.1085  13.9 min
  row 6 anyup128-decoder      281.3 k params | MDE   1384 m (0 no front) | 4-class pixel mIoU 0.762 | 15.9 min

===== heads dinov3-l-sat__caffe__tile512__L6+12+18+24 on v

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

remote code verified: 26 files == reviewed copy (nvidia/C-RADIOv4-H@0057b339059c)


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

cradio-v4-h: 651.6 M parameters loaded
restricted to splits ('train', 'val'): 559 of 681 images
  50/559  0.4 min elapsed, 0.51 s/image, ETA 4.3 min
  100/559  0.7 min elapsed, 0.44 s/image, ETA 3.4 min
  150/559  0.9 min elapsed, 0.37 s/image, ETA 2.5 min
  200/559  1.4 min elapsed, 0.43 s/image, ETA 2.6 min
  250/559  2.0 min elapsed, 0.49 s/image, ETA 2.5 min
  300/559  2.7 min elapsed, 0.53 s/image, ETA 2.3 min
  350/559  3.3 min elapsed, 0.56 s/image, ETA 2.0 min
  400/559  3.6 min elapsed, 0.54 s/image, ETA 1.4 min
  450/559  3.8 min elapsed, 0.51 s/image, ETA 0.9 min
  500/559  3.9 min elapsed, 0.47 s/image, ETA 0.5 min
  550/559  4.3 min elapsed, 0.46 s/image, ETA 0.1 min
  559/559  4.4 min elapsed, 0.47 s/image, ETA 0.0 min
in memory: train X (102263, 1280) (0.3 GB fp16), 121 val images
features ready in 4.4 min | 4000 train tiles kept | RAM used 61 of 179 GB
    step  1000  loss 0.3589  1.2 min
    step  2000  loss 0.1736  2.5 min
    step  3000  loss 0.1442  3.7 min
    step

### Reading the SI table

- Row 1 vs the notebook-06 probe: what the layer choice alone bought.
- Row 2 vs row 1: sharper features, no learning. Row 3 vs row 1: a learned head.
  Row 4: both. The MDE gaps say which one the front task needed.
- `no_front` matters as much as MDE: a head that draws no front on many images
  is not better because its MDE is over an easier subset.

**Run the next cell once only**, after this table has been read and the
decision "which rows go to the test set" is recorded before the run (see README, Protocol). Recorded
decision: all rows of both encoders (C-RADIO and DINOv3 SAT, rows 1-6) plus row 3
seeds 1-2 went to the one test run on 2026-09-25, no selection on SI.

In [6]:
# --- cell: ONE-TIME test run of the trained heads on COL + Mapple (about 1.5 h for both encoders, from the printed step times) ---
import glob
import pandas as pd
from sartransfer.heads import run_heads, summary_table

for encoder, norm in [(e, "caffe") for e in LAYER]:
    try:
        run_heads(encoder, norm, LAYER[encoder], df, DATA, RES, token=os.environ["HF_TOKEN"],
                  anyup=anyup, split="test", rows=(1, 2, 3, 4, 6))
        run_heads(encoder, norm, TAPS[encoder], df, DATA, RES, token=os.environ["HF_TOKEN"],
                  anyup=anyup, split="test", rows=(5,), train_frac=0.01, max_train_tiles=2000)
        run_heads(encoder, norm, LAYER[encoder], df, DATA, RES, token=os.environ["HF_TOKEN"],
                  anyup=anyup, split="test", rows=(3,), seeds=(1, 2), train_frac=0.01)
    except Exception as e:
        print(f"FAILED {encoder}: {type(e).__name__}: {str(e)[:300]}")

te = summary_table(RES, "test")
print("=== TEST (COL + Mapple): MDE and 4-class pixel mIoU per head ===")
print(te[["run", "row", "seed", "head", "params", "MDE_m", "no_front", "mIoU4_pixel"]].round(3).to_string(index=False))
mde = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(str(RES / "heads_test_mde__*.csv")))])
print("\n=== TEST MDE per glacier (m) ===")
print(mde[mde.grouping == "glacier"].pivot_table(index="run", columns="group", values="MDE_m").round(0))
print("\nreference on TEST (step 9): U-Net 979 m (COL 1,080 / Mapple 474); last-layer probes C-RADIO 2,089, photo DINOv3 2,199, DINOv3 SAT 2,805 m; "
      "published baseline COL 840, Mapple 150 m")


===== heads dinov3-l-sat__caffe__tile512__L21 on test: rows [1, 2, 3, 4, 6], seeds [0] =====


Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

dinov3-l-sat: 303.1 M parameters loaded
restricted to splits ('train', 'test'): 560 of 681 images
  50/560  0.5 min elapsed, 0.61 s/image, ETA 5.2 min
  100/560  0.6 min elapsed, 0.34 s/image, ETA 2.6 min
  150/560  0.7 min elapsed, 0.29 s/image, ETA 2.0 min
  200/560  1.2 min elapsed, 0.35 s/image, ETA 2.1 min
  250/560  1.3 min elapsed, 0.30 s/image, ETA 1.6 min
  300/560  1.6 min elapsed, 0.31 s/image, ETA 1.3 min
  350/560  2.1 min elapsed, 0.36 s/image, ETA 1.2 min
  400/560  2.6 min elapsed, 0.39 s/image, ETA 1.0 min
  450/560  3.1 min elapsed, 0.42 s/image, ETA 0.8 min
  500/560  3.5 min elapsed, 0.42 s/image, ETA 0.4 min
  550/560  3.8 min elapsed, 0.41 s/image, ETA 0.1 min
  560/560  3.8 min elapsed, 0.41 s/image, ETA 0.0 min
in memory: train X (2148307, 1024) (4.4 GB fp16), 122 test images
features ready in 3.8 min | 4000 train tiles kept | RAM used 66 of 179 GB
  row 1 linear-grid           4.1 k params | MDE   2660 m (42 no front) | 4-class pixel mIoU 0.677 | 2.4 min
  row 

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

dinov3-l-sat: 303.1 M parameters loaded
restricted to splits ('train', 'test'): 560 of 681 images
  50/560  1.2 min elapsed, 1.44 s/image, ETA 12.3 min
  100/560  1.3 min elapsed, 0.81 s/image, ETA 6.2 min
  150/560  1.7 min elapsed, 0.68 s/image, ETA 4.6 min
  200/560  2.8 min elapsed, 0.84 s/image, ETA 5.0 min
  250/560  3.0 min elapsed, 0.73 s/image, ETA 3.8 min
  300/560  3.7 min elapsed, 0.74 s/image, ETA 3.2 min
  350/560  5.0 min elapsed, 0.85 s/image, ETA 3.0 min
  400/560  6.2 min elapsed, 0.93 s/image, ETA 2.5 min
  450/560  7.5 min elapsed, 1.00 s/image, ETA 1.8 min
  500/560  8.4 min elapsed, 1.00 s/image, ETA 1.0 min
  550/560  9.0 min elapsed, 0.98 s/image, ETA 0.2 min
  560/560  9.0 min elapsed, 0.96 s/image, ETA 0.0 min
in memory: train X (102273, 4096) (0.8 GB fp16), 122 test images
features ready in 9.0 min | 2000 train tiles kept | RAM used 78 of 179 GB
  row 5 grid-decoder-4L      839.3 k params | MDE   1249 m (7 no front) | 4-class pixel mIoU 0.708 | 1.6 min

=====

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

remote code verified: 26 files == reviewed copy (nvidia/C-RADIOv4-H@0057b339059c)


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

cradio-v4-h: 651.6 M parameters loaded
restricted to splits ('train', 'test'): 560 of 681 images
  50/560  0.6 min elapsed, 0.76 s/image, ETA 6.5 min
  100/560  0.7 min elapsed, 0.43 s/image, ETA 3.3 min
  150/560  0.9 min elapsed, 0.36 s/image, ETA 2.4 min
  200/560  1.5 min elapsed, 0.44 s/image, ETA 2.6 min
  250/560  1.6 min elapsed, 0.38 s/image, ETA 2.0 min
  300/560  1.9 min elapsed, 0.39 s/image, ETA 1.7 min
  350/560  2.6 min elapsed, 0.44 s/image, ETA 1.5 min
  400/560  3.2 min elapsed, 0.48 s/image, ETA 1.3 min
  450/560  3.8 min elapsed, 0.51 s/image, ETA 0.9 min
  500/560  4.3 min elapsed, 0.51 s/image, ETA 0.5 min
  550/560  4.6 min elapsed, 0.50 s/image, ETA 0.1 min
  560/560  4.6 min elapsed, 0.49 s/image, ETA 0.0 min
in memory: train X (102263, 1280) (0.3 GB fp16), 122 test images
features ready in 4.6 min | 4000 train tiles kept | RAM used 67 of 179 GB
  row 3 grid-decoder   s1   569.0 k params | MDE   1003 m (9 no front) | 4-class pixel mIoU 0.680 | 1.3 min
  row 3 g

### Precision check (code review 2026-09-25) -- SI only, saved heads, no training

Every head ran under bf16 autocast, and AnyUp then builds its pixel positions
and RoPE angles in bf16 (at 512 px only 385 of 512 positions stay distinct).
This cell loads the **saved** heads of rows 2, 4 and 6 and runs one tile per SI
image at fp32, at bf16 as run, and at bf16 with fp32 positions. The test
glaciers are not touched.

**Decision rule, fixed before the numbers:** re-run the AnyUp rows at full
precision on SI only if the as-run variant changes the class of
more than 1 % of valid pixels in row 2, 4 or 6, or its feature error exceeds
10 % of AnyUp's own effect. Needs cells 1-3 first. About 0.5 min per encoder once the model is loaded.

In [4]:
# --- cell: PRECISION CHECK -- AnyUp fp32 vs bf16 on SI tiles, saved heads, no training (0.5 min per encoder after loading) ---
from sartransfer.checks import anyup_precision_check

for encoder, layer in (("dinov3-l-sat", 21), ("cradio-v4-h", 32)):
    try:
        anyup_precision_check(encoder, "caffe", layer, df, DATA, RES,
                              token=os.environ["HF_TOKEN"], anyup=anyup)
    except Exception as e:
        print(f"FAILED {encoder}: {type(e).__name__}: {str(e)[:300]}")


===== AnyUp precision check dinov3-l-sat__caffe__tile512__L21: 12 SI images, one tile each =====


Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

dinov3-l-sat: 303.1 M parameters loaded


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:2954: UserWarning: Mismatch dtype between input and weight: input dtype = c10::BFloat16, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at /pytorch/aten/src/ATen/native/layer_norm.cpp:344.)
  return torch.rms_norm(input, normalized_shape, weight, eps)


                       feat_rel_err  anyup_effect  row2_changed_pct  row4_changed_pct  row6_changed_pct
variant                                                                                                
fp32                         0.0000        0.4074            0.0000            0.0000            0.0000
bf16 (as run)                0.0100        0.4074            0.3736            0.1504            0.1319
bf16 + fp32 positions        0.0092        0.4074            0.3396            0.1266            0.1125
as run: feature error = 2.4 % of AnyUp's own effect (rule: > 10 %); up to 0.37 % of valid pixels change class (rule: > 1 %) -> precision does not drive the AnyUp result at inference | 0.5 min

===== AnyUp precision check cradio-v4-h__caffe__tile512__L32: 12 SI images, one tile each =====
installing ['open_clip_torch']


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

remote code verified: 26 files == reviewed copy (nvidia/C-RADIOv4-H@0057b339059c)


Loading weights:   0%|          | 0/390 [00:02<?, ?it/s]

cradio-v4-h: 651.6 M parameters loaded
                       feat_rel_err  anyup_effect  row2_changed_pct  row4_changed_pct  row6_changed_pct
variant                                                                                                
fp32                         0.0000        0.3087            0.0000            0.0000            0.0000
bf16 (as run)                0.0077        0.3087            0.1983            0.1085            0.1252
bf16 + fp32 positions        0.0068        0.3087            0.1757            0.0918            0.1179
as run: feature error = 2.5 % of AnyUp's own effect (rule: > 10 %); up to 0.20 % of valid pixels change class (rule: > 1 %) -> precision does not drive the AnyUp result at inference | 0.5 min
